# Notebook 01: Feature Engineering Fundamentals

## Why This Matters

Feature engineering is the craft of transforming raw data into representations that ML algorithms can learn from effectively — it is arguably the single highest-leverage skill in applied ML. Classical ML interviews almost always include a feature engineering round, because real-world data is messy and a good model on bad features will always lose to a mediocre model on great features. Understanding the fundamentals — what a feature is, how scale and distribution affect models, and how to systematically explore a dataset — separates practitioners from people who only know the API. These concepts also underpin every deep-learning preprocessing pipeline, making this knowledge durable across the entire field.

## 1. What Is Feature Engineering?

Feature engineering is the process of using domain knowledge to select, transform, and create input variables (features) that make supervised or unsupervised learning algorithms work better. Raw data almost never arrives in the form a model needs: timestamps must become hour-of-day integers, IP addresses must become network-range flags, free text must become token counts. The goal is to encode the signal that matters while suppressing irrelevant variation.

Key vocabulary:
- **Feature / predictor / covariate**: a single measurable property of an observation
- **Feature space**: the n-dimensional space defined by n features
- **Target leakage**: accidentally encoding information about the label into a feature (train AUC=0.99, production AUC=0.51)
- **Feature drift**: the distribution of a feature changes between training and serving time

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import fetch_california_housing

# Load a real dataset to motivate examples throughout the notebook
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print("Shape:", df.shape)
print("\nFeature descriptions:")
for feat, desc in zip(housing.feature_names, housing.feature_names):
    print(f"  {feat}")
df.head(3)

## 2. Exploratory Data Analysis as a Feature Engineering Tool

Before building any features you must understand the raw data. The EDA phase answers: What is the distribution of each feature? Are there outliers? What is the relationship between each feature and the target? Which features are correlated with each other (multicollinearity)? EDA is not optional; skipping it is how practitioners ship bugs that corrupt entire production systems.

The three most useful EDA operations before feature engineering:
1. **`.describe()`** — spot anomalous min/max values, near-zero variance
2. **Histogram grid** — understand shape (normal, skewed, bimodal, uniform)
3. **Correlation heatmap** — find multicollinearity and candidate interaction terms

In [ ]:
print("=== Descriptive Statistics ===")
print(df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
axes = axes.ravel()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=50, edgecolor='k', alpha=0.7)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('Count')
plt.suptitle('Feature Distributions — California Housing', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print("Observation: AveRooms, AveBedrms, AveOccup, Population are heavily right-skewed — log transform will help linear models.")

In [ ]:
import seaborn as sns

corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — California Housing', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Scale and Distribution: Why It Matters for Models

Many algorithms are sensitive to the scale and distribution of features:
- **Distance-based models** (KNN, SVM, K-Means): A feature measured in thousands dominates features measured in single digits unless you normalize.
- **Gradient descent** (logistic regression, neural nets): Large feature ranges cause the loss surface to be elongated, slowing convergence.
- **Tree-based models** (Decision Tree, Random Forest, XGBoost): Split on thresholds, so scale does not matter — but heavy skew can still reduce split quality.
- **Regularized models** (Ridge, Lasso): Penalties are applied to coefficients in feature-space units; unscaled features make the penalty asymmetric.

The three standard scalers and when to use them:

| Scaler | Formula | Use when |
|--------|---------|----------|
| StandardScaler | (x - μ) / σ | Feature is approximately Gaussian, no extreme outliers |
| MinMaxScaler | (x - min) / (max - min) | Bounded range needed (e.g., neural net input), no outliers |
| RobustScaler | (x - median) / IQR | Feature has outliers you want to shrink but not drop |

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Demonstrate on the Population feature (heavily right-skewed with outliers)
pop = df[['Population']].copy()

scalers = {
    'Original': pop.values,
    'StandardScaler': StandardScaler().fit_transform(pop),
    'MinMaxScaler': MinMaxScaler().fit_transform(pop),
    'RobustScaler': RobustScaler().fit_transform(pop),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (name, vals) in zip(axes, scalers.items()):
    ax.hist(vals, bins=60, edgecolor='k', alpha=0.7)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Effect of Scaling on Population Feature', fontsize=12)
plt.tight_layout()
plt.show()

# Summary stats after scaling
for name, vals in scalers.items():
    v = vals.ravel()
    print(f"{name:15s} | mean={v.mean():7.3f}  std={v.std():6.3f}  min={v.min():8.3f}  max={v.max():8.3f}")

## 4. Transforming Skewed Features

Right-skewed features (long tail to the right) violate the Gaussian assumption in linear models and make gradient descent harder. The standard toolkit:

- **Log transform**: `log(x + 1)` — works great for count data and monetary values; cannot handle negatives or zeros without the +1 offset.
- **Square-root transform**: `sqrt(x)` — milder compression than log; handles zeros naturally.
- **Box-Cox**: parametric family that finds the optimal power λ; requires all positive values.
- **Yeo-Johnson**: like Box-Cox but handles zero and negative values; safe default.

Interview tip: state which transform you'd choose and *why* (data constraints + distribution shape), not just that you'd "apply a transform."

In [ ]:
from sklearn.preprocessing import PowerTransformer

pop_vals = df['Population'].values.reshape(-1, 1)

transforms = {
    'Original': pop_vals.ravel(),
    'log1p': np.log1p(pop_vals.ravel()),
    'sqrt': np.sqrt(pop_vals.ravel()),
    'Yeo-Johnson': PowerTransformer(method='yeo-johnson').fit_transform(pop_vals).ravel(),
    'Box-Cox': PowerTransformer(method='box-cox').fit_transform(pop_vals).ravel(),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, vals) in zip(axes, transforms.items()):
    ax.hist(vals, bins=60, edgecolor='k', alpha=0.75)
    skew = pd.Series(vals).skew()
    ax.set_title(f"{name}\nskew={skew:.2f}", fontsize=9)
plt.suptitle('Skewness Reduction Transforms — Population Feature', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Interaction Features and Polynomial Features

Linear models can only learn linear decision boundaries by default. Adding **interaction terms** (e.g., `rooms_per_person = AveRooms / AveOccup`) or **polynomial features** (e.g., `MedInc²`) lets linear models capture non-linear relationships without switching to a more complex algorithm.

Trade-off: polynomial features grow as O(n^d) where n is the number of features and d is the degree. With 100 features at degree 2 you get ~5000 features — regularization becomes essential, and interpretability suffers.

In interviews, domain-informed interactions (e.g., price-per-sqft, clicks-per-impression) are valued over blind polynomial expansion.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# Domain-informed feature engineering
df_eng = df.copy()
df_eng['RoomsPerPerson']   = df['AveRooms'] / df['AveOccup']
df_eng['BedroomRatio']     = df['AveBedrms'] / df['AveRooms']
df_eng['log_Population']   = np.log1p(df['Population'])
df_eng['log_AveOccup']     = np.log1p(df['AveOccup'])

base_features = housing.feature_names
eng_features  = list(df_eng.columns.drop('MedHouseVal'))
y = df_eng['MedHouseVal']

for feat_list, label in [(base_features, 'Raw features'), (eng_features, 'Engineered features')]:
    X = df_eng[feat_list]
    pipe = Pipeline([('scale', StandardScaler()), ('lr', LinearRegression())])
    scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    print(f"{label:25s} | R² = {scores.mean():.4f} ± {scores.std():.4f}  (n_features={len(feat_list)})")

In [ ]:
# sklearn PolynomialFeatures — degree 2 on a small subset
X_small = df[['MedInc', 'HouseAge']].values
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_small)
print("Original features   :", poly.n_features_in_)
print("Polynomial features :", X_poly.shape[1])
print("Feature names       :", poly.get_feature_names_out())

## 6. Target Leakage: The Silent Model Killer

Target leakage occurs when a feature includes information about the label that would not be available at prediction time. Classic examples:
- Predicting loan default with "days_delinquent" (only nonzero *after* default begins)
- Predicting churn with "support_tickets_after_churn_date"
- Using post-treatment data to predict treatment outcome

Leakage produces unrealistically high cross-validation scores that evaporate in production. The fix is strict temporal and causal discipline: features must only encode information that was observable *before* the outcome event.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Simulate leakage: a "leaked" feature is just label + noise
np.random.seed(42)
n = 2000
X_real = np.random.randn(n, 5)
y_true = (X_real[:, 0] + 0.5 * X_real[:, 1] > 0).astype(int)

# Leaked feature: strongly correlated with label but not available at serving time
leaked = y_true + 0.1 * np.random.randn(n)  # essentially the label

X_train, X_test, y_train, y_test = train_test_split(X_real, y_true, test_size=0.3, random_state=42)
X_leak_train = np.column_stack([X_real[:700], leaked[:700]])
X_leak_test  = np.column_stack([X_real[700:], leaked[700:]])  # WRONG: leaked at test time too

for label, Xtr, Xte in [
    ('No leakage', X_train, X_test),
    ('With leakage (train+test)', X_leak_train, X_leak_test),
]:
    lr = LogisticRegression(max_iter=200).fit(Xtr, y_train)
    auc = roc_auc_score(y_test, lr.predict_proba(Xte)[:, 1])
    print(f"{label:35s} | AUC = {auc:.4f}")

## Summary

| Concept | Core Idea | Interview Tip |
|---------|-----------|---------------|
| Feature engineering | Transform raw data into algorithm-ready signals | Always mention domain knowledge as the primary driver |
| EDA first | Understand distributions, outliers, correlations before transforming | Name `.describe()`, histogram grid, correlation heatmap |
| Scaling | Standardize for gradient/distance models; trees are scale-invariant | Justify scaler choice by data characteristics |
| Skew transforms | log1p, sqrt, Yeo-Johnson reduce skew for linear models | Box-Cox requires positive values; Yeo-Johnson does not |
| Interaction features | Encode domain logic as ratios, products, differences | Better than blind polynomial expansion |
| Target leakage | Post-outcome features inflate training metrics | Fix with strict temporal ordering; detect via suspiciously high AUC |